# Experimento A/B en página de inicio

El objetivo de este proyecto es evaluar un **experimento A/B** realizado en una página de inicio (landing page) con versiónes **A y B** para apoyar una **decisión de negocio basada en datos**.

---

El archivo `landing_experiment.csv` contiene información de usuarios expuestos a dos versiones de la página de inicio (landing page) dentro del experimento A/B. Incluye región, dispositivo, fuente de tráfico, tipo de usuario, conversión y gasto.

El análisis sigue una lógica clara y progresiva:

1. 🔍 Explorar y validar los datos.

2. 💰 Comparar el **gasto promedio** por usuario entre la página A y B.

3. 🎯 Comparar la **tasa de conversión** entre la página A y B.

4. 🌐 Revisar **la relación entre la fuente de tráfico y la conversión**.

5. 👤 Revisar **la relación entre el tipo de usuario y la conversión**.

6. 📈 **Visualizar los resultados**: Respalda tus conclusiones mediante gráficos claros.

Se aplican **puebas estadísticas apropiados** para comparar las páginas y **recomendar qué versión es mejor**, justificando la decisión con datos.

## 🧩 Paso 1: Cargar y validar los datos

### 1.1 Carga de datos y vista rápida

In [37]:
# importar librerías
import pandas as pd
from scipy.stats import ttest_ind, levene
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import chi2_contingency
import seaborn as sns
import matplotlib.pyplot as plt

#Asignación de variables fijas
ALPHA = 0.05

In [38]:
# cargar archivo
df_landing = pd.read_csv('/datasets/landing_experiment.csv')

#Se crea una copia para tener los datos originales
df_landing_copy = df_landing.copy()

**Vista previa e información general del conjunto de datos**

In [39]:
# mostrar las primeras 5 filas

print('Tabla muestra del dataset')
df_landing.head()

Tabla muestra del dataset


,user_id,date,landing,region,dispositivo,traffic_source,user_type,converted,gasto
0,26f3052e-8500-44ea-8fff-06de65258abb,2026-01-01,A,Norte,Mobile,Email,Recurrente,1,38.08
1,92378c09-4bbf-40c7-945e-82b84f392d22,2026-01-23,A,Occidente,Mobile,Organic,Nuevo,0,0.00
2,a4397360-40e5-45d6-a7ff-dcb4da2c9a1f,2026-01-01,B,Centro,Mobile,Organic,Nuevo,0,0.00
3,7ca3a26f-1e6c-44aa-9b09-b8cb01112956,2026-01-22,A,Centro,Mobile,Ads,Nuevo,0,0.00
4,8dc9593b-5b9c-479d-848b-a99493920419,2026-01-16,A,Sur,Mobile,Organic,Nuevo,0,0.00


In [40]:
# información general
print('Información general')
print(df_landing.info())
print()
print('Lista de conteo de valores únicos')
print(df_landing.nunique().sort_values(ascending=False))
print()

Información general
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   user_id         40000 non-null  object 
 1   date            40000 non-null  object 
 2   landing         40000 non-null  object 
 3   region          40000 non-null  object 
 4   dispositivo     40000 non-null  object 
 5   traffic_source  40000 non-null  object 
 6   user_type       40000 non-null  object 
 7   converted       40000 non-null  int64  
 8   gasto           40000 non-null  float64
dtypes: float64(1), int64(1), object(7)
memory usage: 2.7+ MB
None

Lista de conteo de valores únicos
user_id           40000
gasto              4229
date                 28
region                5
traffic_source        4
landing               2
dispositivo           2
user_type             2
converted             2
dtype: int64



✍️ **Comentario**:

    - Los datos de la columna `date` deben ser del tipo datetime no object.

 Se aplica corrección en la siguiente línea


In [41]:
# corrección de la columna date
df_landing['date'] = pd.to_datetime(df_landing['date'], errors='coerce')

#Información general
print('Información general')
print(df_landing.info())
print()
print('Lista de conteo de valores únicos')
print(df_landing.nunique().sort_values(ascending=False))
print()

Información general
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   user_id         40000 non-null  object        
 1   date            40000 non-null  datetime64[ns]
 2   landing         40000 non-null  object        
 3   region          40000 non-null  object        
 4   dispositivo     40000 non-null  object        
 5   traffic_source  40000 non-null  object        
 6   user_type       40000 non-null  object        
 7   converted       40000 non-null  int64         
 8   gasto           40000 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(6)
memory usage: 2.7+ MB
None

Lista de conteo de valores únicos
user_id           40000
gasto              4229
date                 28
region                5
traffic_source        4
landing               2
dispositivo           2
user_type  

**Descripción del conjunto de datos**

El dataset contiene las siguientes columnas:

- `user_id` — Identificador único del usuario
- `date` — Fecha en la que el usuario fue expuesto a la página
- `landing` — Versión de la página mostrada al usuario
- `region` — Región geográfica del usuario
- `dispositivo` — Tipo de dispositivo utilizado por el usuario
- `traffic_source` — Canal por el que llegó el usuario
- `user_type` — Tipo de usuario según historial previo
- `converted` — Indica si el usuario realizó una conversión
- `gasto` — Monto gastado por el usuario (0 si no convirtió)

### 1.2 Análisis exploratorio y revisión de calidad de datos

Se identifican las variables clave del experimento A/B y se valida que estén bien definidas, completas y que sean consistentes.


In [42]:
#Previamente se crea una función para clasificar las columnas

def clasificar_columnas(df):
    """Clasifica columnas en numéricas, booleanas y categóricas."""

    columnas_ignorar = ['user_id', 'date']

    numericas, booleanas, categoricas = [], [], []

    for col in df.columns:

        #Condición para ignorar columnas
        if col in columnas_ignorar:
            continue

        valores_unicos = df[col].dropna().nunique()
        if valores_unicos <= 2:
            booleanas.append(col)
        elif pd.api.types.is_numeric_dtype(df[col]):
            numericas.append(col)
        else:
            categoricas.append(col)

    return list(df.columns), numericas, booleanas, categoricas

# Llamada única — resultado reutilizado en todas las celdas siguientes

todas_columnas, numericas, booleanas, categoricas = clasificar_columnas(df_landing)

# Tabla de clasificación con dict comprehension (un solo bucle)
mapeo_clasificacion = (
    {col: 'Numerica'   for col in numericas}
    | {col: 'Booleana'  for col in booleanas}
    | {col: 'Categórica' for col in categoricas}
)

tabla_variables = (
    pd.DataFrame(mapeo_clasificacion.items(), columns=['Columna', 'Tipo de variable'])
    .sort_values('Tipo de variable')
    .reset_index(drop=True)  # drop=True evita columna de índice redundante
)

print('Clasificación de las columnas')
display(tabla_variables)

Clasificación de las columnas


,Columna,Tipo de variable
0,landing,Booleana
1,dispositivo,Booleana
2,user_type,Booleana
3,converted,Booleana
4,region,Categórica
5,traffic_source,Categórica
6,gasto,Numerica


 **Variable `user_id`**  
 Verificar usuarios únicos

In [43]:
df_conteo_duplicados = df_landing['user_id'].duplicated().sum()
if df_conteo_duplicados == 0:
    print(f"Resultado: {df_conteo_duplicados} duplicados")
    print("Conjunto de datos aptos para aplicar las pruebas estadísticas")
else:
    print("Hay valores duplicados, por tanto no aplica prueba estadística chi-cuadrado, porque no cumple el primer criterio")


Resultado: 0 duplicados
Conjunto de datos aptos para aplicar las pruebas estadísticas


 **Variable `date`**  
Explorar rango de fechas

In [44]:
# Resumen estadístico
df_landing["date"].describe()

count                   40000
unique                     28
top       2026-01-24 00:00:00
freq                     1512
first     2026-01-01 00:00:00
last      2026-01-28 00:00:00
Name: date, dtype: object

In [45]:
# Identificar rango temporal del experimento
print("Fecha mínima:", df_landing["date"].min())
print("Fecha máxima:", df_landing["date"].max())

Fecha mínima: 2026-01-01 00:00:00
Fecha máxima: 2026-01-28 00:00:00


**Variable `gasto` (numérica)**

In [46]:
# Resumen estadístico
df_landing[numericas].describe()

,gasto
count,40000.000000
mean,9.325554
std,25.667986
min,0.000000
25%,0.000000
50%,0.000000
75%,0.000000
max,303.680000


In [47]:
# Resumen estadístico de usuarios que se convirtieron
df_landing[booleanas].describe(include='all')

,landing,dispositivo,user_type,converted
count,40000,40000,40000,40000.00000
unique,2,2,2,NaN
top,B,Mobile,Nuevo,NaN
freq,20018,24829,26033,NaN
mean,NaN,NaN,NaN,0.14265
std,NaN,NaN,NaN,0.34972
min,NaN,NaN,NaN,0.00000
25%,NaN,NaN,NaN,0.00000
50%,NaN,NaN,NaN,0.00000
75%,NaN,NaN,NaN,0.00000


 **Variables categóricas**  
 Verificar categorías esperadas del experimento ( A y B).

In [48]:
# Explorar variables categóricas y cómo se distribuyen
print("\nConteo de categorías:")
print(df_landing[categoricas].describe())


Conteo de categorías:
       region traffic_source
count   40000          40000
unique      5              4
top     Norte        Organic
freq    11166          17987


✍️ **Comentario**:

    - Todas las columnas tienen valores esperados.


## 💰 Paso 2: Comparar el gasto promedio por usuario (página A vs B)

Se evalua si existen diferencias estadísticamente significativas en el gasto promedio de los **usuarios que se convirtieron en clientes** entre la página A y la página B, para identificar qué versión genera **mayor valor económico** para el negocio.


In [49]:
# Filtración base por página de prueba que si compraron
conversion_landing_a = (df_landing['landing']=='A') & (df_landing['converted']==1)
conversion_landing_b = (df_landing['landing']=='B') & (df_landing['converted']==1)

# Gasto por versión
gasto_A = df_landing[conversion_landing_a]['gasto']
gasto_B = df_landing[conversion_landing_b]['gasto']

# Verificar cantidad de datos que tiene cada grupo
print("Total de conversión por página")

print(f" A: {len(gasto_A)}")
print(f" B: {len(gasto_B)}")

Total de conversión por página
 A: 2512
 B: 3194


### Prueba estadística t-test para comparar promedios

**Hipótesis:**
- **Hipótesis nula (H₀):** El gasto promedio es igual en ambas paginas.
- **Hipótesis alternativa (H₁):** El gasto promedio es diferente entre las paginas.

#### Creación de la función t-test

In [50]:

def prueba_estadistica_t_test(col1, col2, nombre_col1='grupo_1', nombre_col2='grupo_2'):
    """Aplica Levene + t-Welch o t-Student según igualdad de varianzas.

    Parámetros
    ----------
    lista1 : array-like — Asignación de la primera columna
    lista2 : array-like — Asignación de la segunda columna
    lista3 : array-like — Asignación del nombre de la primera columna, colocar comilla sencillo o doble
    lista4 : array-like — Asignación del nombre de la segunda columna, colocar comilla sencillo o doble
    """

    def _imprimir_promedios():
        """Bloque de promedios reutilizable (antes duplicado en ambas ramas)."""
        media_1 = round(col1.mean(), 2)
        media_2 = round(col2.mean(), 2)
        print('Rechazo de la hipótesis nula: hay una evidencia de una diferencia')
        print()
        print('Promedios')
        print(f'  {nombre_col1} : {media_1}')
        print(f'  {nombre_col2} : {media_2}')
        print(f'  diferencia_media : {round(media_1 - media_2, 2)}')

    # Prueba de Levene (igualdad de varianzas)
    l_stat, p_value_var = levene(col1, col2)
    print('Resultado de la prueba estadística de levene (varianzas iguales)')
    print(f'  Estadístico de levene: {l_stat}')
    print(f'  Valor p: {p_value_var}')
    print()

    varianzas_iguales = p_value_var > ALPHA   # True → t-Student / False → t-Welch
    nombre_prueba = 't-test' if varianzas_iguales else 't-welch'

    if varianzas_iguales:
        print('Conclusión: No rechazamos H₀ de varianzas → equal_var=True (t-Student)')
    else:
        print('Conclusión: Rechazamos H₀ de varianzas → equal_var=False (t-Welch)')

    # Prueba t (rama única, parámetro equal_var determinado arriba)
    t_stat, p_value = ttest_ind(col1, col2, equal_var=varianzas_iguales)
    print()
    print(f'Resultado de la prueba estadística {nombre_prueba}')
    print(f'  Estadístico t: {t_stat}')
    print(f'  Valor p: {p_value}')
    print()
    print('Conclusión:')

    if p_value <= ALPHA:
        _imprimir_promedios()
    else:
        print('No se rechaza la hipótesis nula: no hay evidencia de una diferencia')

#### Aplicar prueba t-test

In [51]:

prueba_estadistica_t_test(gasto_A, gasto_B, 'gasto a', 'gasto b')


Resultado de la prueba estadística de levene (varianzas iguales)
  Estadístico de levene: 29.17646453202917
  Valor p: 6.875301988016449e-08

Conclusión: Rechazamos H₀ de varianzas → equal_var=False (t-Welch)

Resultado de la prueba estadística t-welch
  Estadístico t: -9.48101092267275
  Valor p: 3.627602231521493e-21

Conclusión:
Rechazo de la hipótesis nula: hay una evidencia de una diferencia

Promedios
  gasto a : 61.09
  gasto b : 68.75
  diferencia_media : -7.66



### 📝 Conclusión e interpretación

**Decisión:**  
Se rechaza la hipótesis nula  con un nivel de confianza superior al 99%.

**Interpretación de negocio:**  

Existe evidencia estadísticamente significativa de que hay un incremento del 7.66 unidades monetarias, representando el 12.54% de la página B con respecto a la página A.

Este resultado sugiere que la página B puede tener un impacto positivo en los ingresos.


---


## 📈 Paso 3: Comparar la tasa de conversión entre la página A y B

Se evalua si existen diferencias estadísticamente significativas en la **tasa de conversión** entre la página A y B, con el fin de identificar qué versión genera **mayor número de usuarios convertidos**.

### Prueba estadística z-test para proporciones

**Hipótesis:**
- **Hipótesis nula (H₀):** La tasa de conversión es igual en ambas páginas.
- **Hipótesis alternativa (H₁):** La tasa de conversión es diferente entre ambas páginas.

#### Creación de la función z-test

In [52]:

def prueba_z_test(exitos_n, observaciones_n, elemento_a='elemento_a', elemento_b='elemento_b'):
       
        """Prueba z de proporciones.

        Parámetros
        ----------
        lista1 : array-like — conteo de éxitos por grupo [A, B]
        lista2 : array-like — conteo total de observaciones por grupo [A, B]
        """
        z_stat, p_value = proportions_ztest(exitos_n, observaciones_n)

        print('Matriz a usar')
        print('Total conversión:', exitos_n)
        print('Total conteo    :', observaciones_n)
        print()
        print('Resultado de la prueba estadística z-test')
        print(f'  Estadístico z: {z_stat}')
        print(f'  Valor p      : {p_value}')
        print()
        print('Conclusión:')

        if p_value <= ALPHA:
            print('Rechazamos la hipótesis nula: hay una evidencia de una diferencia')

            # Usa los parámetros de la función (antes usaba variables globales — bug corregido)
            tasa_A = exitos_n[0] / observaciones_n[0]
            tasa_B = exitos_n[1] / observaciones_n[1]

            print()
            print('Distribución del porcentaje')
            print(f'  Tasa de conversión {elemento_a}: {tasa_A:.2%}')
            print(f'  Tasa de conversión {elemento_b}: {tasa_B:.2%}')
            print()
            print('Conclusión:')

            if tasa_A > tasa_B:
                print(f'{elemento_a}: tiene una diferencia de {round((tasa_A - tasa_B),4)*100} punto porcentual')
            elif tasa_B > tasa_A:
                print(f'{elemento_b} tiene una diferencia de {round((tasa_B - tasa_A),4)*100} punto porcentual')
            else:
                print('Los elementos evaluados tienen la misma tasa de conversión')
        else:
            print('No rechazamos la hipótesis nula: no hay evidencia suficiente de una diferencia')


#### Aplicación de la función z-test

In [53]:

# Número de usuarios convertidos por página
df_landing_converted = df_landing.groupby('landing')['converted'].sum()

# Total de usuarios por página
df_landing_total = df_landing.groupby('landing')['converted'].count()
print()

# Matriz para usarlo en la prueba t-test
exitos        = [df_landing_converted['A'], df_landing_converted['B']]
observaciones = [df_landing_total['A'], df_landing_total['B']]

#Aplicación de la función z-test
prueba_z_test(exitos, observaciones, 'página A', 'página B')



Matriz a usar
Total conversión: [2512, 3194]
Total conteo    : [19982, 20018]

Resultado de la prueba estadística z-test
  Estadístico z: -9.677362674655983
  Valor p      : 3.7629765627523803e-22

Conclusión:
Rechazamos la hipótesis nula: hay una evidencia de una diferencia

Distribución del porcentaje
  Tasa de conversión página A: 12.57%
  Tasa de conversión página B: 15.96%

Conclusión:
página B tiene una diferencia de 3.38 punto porcentual



### 📝 Conclusión e interpretación

**Decisión:**  
Se rechaza la hipótesis nula con un nivel de confianza superior al 99%.

**Interpretación de negocio:**  
Existe evidencia estadísticamente significativa de que hay un incremento del 26.96% de la página B con respecto a la página A.

Este resultado sugiere que la página B puede tener un impacto positivo en los ingresos.


## 🔗 Paso 4: Revisar la relación entre la fuente de tráfico y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre la **fuente de tráfico** (`traffic_source`) y la **conversión** (`converted`), para identificar qué canales generan más conversiones.

### Prueba Aplicación estadística chi-cuadrada para variables categóricas

**Hipótesis:**
- **Hipótesis nula (H₀):** La conversión es independiente de la fuente.
- **Hipótesis alternativa (H₁):** La conversión depende de la fuente.

#### Creación de la función chi-cuadrado

In [58]:
def prueba_chi_cuadrado(col1, col2, col3, col4,df):

    df_duplicado = col1.duplicated().sum()
    
    if df_duplicado == 0:

        print('Se cumple primer criterio: no hay duplicados, conjunto de datos aptos para prueba chi-cuadrado')
    
        df_conteo = col2.value_counts()

        tabla = pd.crosstab(col3, col4)

        tabla_normalizada = (pd.crosstab(col3, col4, normalize='index')*100).round(2)
        print("\nTabla para la prueba de chi-cuadrado")
        display(tabla)

        chi2_stat, p_value, dof, expected = chi2_contingency(tabla)

        print(f"Estadístico chi-cuadrdo: {chi2_stat:.3f}")
        print(f"Valor P: {p_value:.3f}")

        if p_value < ALPHA:
            print("\nRechazamos la hipótesis nula: hay evidencia de asociación entre las variables.")
            print("\nTabla normalizada")
            display(tabla_normalizada)
    
            print("Frecuencias esperadas")
            print(expected.round(0))

            #Criterio de aceptación: al menos el 20% de los datos obtenidos son menor a 5 
            pct_celdas_invalidas = (expected < 5).sum() / expected.size * 100

            if pct_celdas_invalidas > 20:
                print("\nNo se cumple el criterio:",
                      f"\nAl menos el 20% (resultado: {pct_celdas_invalidas:.1f}) de las frecuencias esperadas son < 5.", 
                      "\nLa prueba chi-cuadrada no es confiable para este conjunto de datos")
            else:
                print("\nSe cumple el criterio:",
                    f"\nAl menos el 80% (Resultado: {100-pct_celdas_invalidas}) de las frecuencias esperadas son >= 5")
                
                #Unión de las dos gráficas con subplots

                fig, axes = plt.subplots(1, 2, figsize=(181,6))

                #Barras agrupadas
                #----------------------------------------------------------------------------
                ax1 = sns.countplot(data = df, x=col3, hue = col4, palette = ["#95D5B2", "#F4A261"], ax=axes[0])
                
                #Agregar valores
                for bar in ax1.patches:                             #Recorrer todas las barras del gráfico
                    height = bar.get_height()
                    
                    #Obtener la altura de la barra (conteo de usuarios)

                    if height > 0:                                       #Asegurar que no se impriman valores pequeños que no correspondan
                        ax1.text(x=bar.get_x()+bar.get_width()/2,        #Establecer coordenada X del texto (centro de la barra)
                                 y=height+100,                           #Establecer coordenada Y del texto (parte superior de la barra con espacio)
                                 s=f"{height:.0F}",                      #Valor numérico a mostrar en entero
                                 ha='center',                            #Alinear centro del texto en coordenada
                                 va='bottom',                            #Alinear parte inferior del texto coordenada
                                 fontsize=11)                            #Tamaño de la fuente
                               
                axes[0].set_title('Conversiones por tipo de dispositivo', fontsize = 15)
                axes[0].set_xlabel('Dispositivo', fontsize = 12)
                axes[0].set_ylabel('Cantidad de usuarios', fontsize = 12)
                #plt.ylim(0,14000)
                axes[0].legend(title = '¿Convirtió?', labels=['No (0)', 'Sí (0)'])
                #plt.show()

                #Barras apiladas
                #----------------------------------------------------------------------------
                ax2 = tabla_normalizada.plot(kind='bar', stacked=True, figsize=(10,6), color = ["#8ecae6", "#95d5b2"], ax=axes[1])

                #Agregar valores
                for bar in ax2.patches:                             #Recorrer todas las barras del gráfico
                    height = bar.get_height()                       #Obtener la altura de la barra (conteo de usuarios)
                    ax2.text(x=bar.get_x()+bar.get_width()/2,       #Establecer coordenada X del texto centrado
                            y=bar.get_y()+height/2,                 #Establecer coordenada Y del texto centrado
                            s=f"{height:.2F}",                      #Valor numérico a mostrar con dos decimales
                            ha='center',                            #Alinear centro del texto en coordenada
                            va='center',                            #Alinear parte centro del texto coordenada
                            fontsize=11)                            #Tamaño de la fuente

                axes[1].set_title('Tasa de conversión por Dispositivo', fontsize=15)
                axes[1].set_ylabel('Porcentaje (%)', fontsize=12)
                axes[1].set_xlabel('Dispositivo', fontsize=12)               
                axes[1].set_xlim(-1,4)
                axes[1].legend(title='¿Convirtió?', labels=['No (0)', 'Sí (1)'], loc= 'upper right')
                axes[1].tick_params(axis='x', rotation= 0)
                plt.tight_layout
                plt.show()
        
        else:
            print("\nNo rechazamos la hipótesis nula: no hay evidencia suficiente de asociación entre las variables.")
    else:
        print('No se cumple la primera condición: si hay duplicados y por ende, la prueba chi-cuadrada no es apta')

#### Aplicación de la función chi-cuadrado


In [60]:

#Selección de la columna de referencia para verificar ausencia de duplicados.
df_duplicado = df_landing[['user_id']]

#Selección de las columnas a comparar.

df_par = df_landing[['traffic_source','converted']]

# Aplicar prueba 
prueba_chi_cuadrado(df_duplicado, df_par, df_landing['traffic_source'], df_landing['landing'],df_landing)


Se cumple primer criterio: no hay duplicados, conjunto de datos aptos para prueba chi-cuadrado

Tabla para la prueba de chi-cuadrado


landing,A,B
traffic_source,,
Ads,5981,5954
Email,3087,3036
Organic,8992,8995
Referral,1922,2033


Estadístico chi-cuadrdo: 3.569
Valor P: 0.312

No rechazamos la hipótesis nula: no hay evidencia suficiente de asociación entre las variables.


### 📝 Conclusión e interpretación

**Decisión:**  
No se rechaza la hipótesis nula

**Interpretación de negocio:**  
El resultado del p value obtenido es 0.312 y es mayor al ALPHA = 0.05, por tanto, esta relación no es relevante para el negocio.


## 👤 Paso 5: Revisar la relación entre el tipo de usuario y la conversión

Se analiza si existe una **asociación estadísticamente significativa** entre el **tipo de usuario** (`user_type`) y la **conversión** (`converted`), entendiendo que un usuario recurrente puede haber visitado antes sin necesariamente convertirse en cliente en esta ocasión.

El objetivo es identificar qué perfiles muestran mayor probabilidad de conversión dentro del contexto analizado.

### Prueba ...

**Hipótesis:**
- **Hipótesis nula (H₀):** ...
- **Hipótesis alternativa (H₁):** ...

In [ ]:
# Aplicar prueba



### 📝 Conclusión e interpretación

**Decisión:**  
(¿Se rechaza o no la hipótesis nula?)

**Interpretación de negocio:**  
Explica qué indica el resultado.

## 📊 Paso 6: Visualizar los resultados de variables categóricas

Se explora visualmente la relación entre variables categóricas (`traffic_source` y `user_type`) y la conversión, mostrando para cada categoría:
- la cantidad absoluta de usuarios que convirtieron y no convirtieron,
- la proporción de usuarios que convirtieron y no convirtieron.

Esto permite analizar tanto el impacto en volumen como la efectividad relativa de cada categoría y reforzar los resultados obtenidos en las pruebas estadísticas.

### Relación entre la fuente de tráfico y la conversión

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

### Relación entre el tipo de usuario y la conversión

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

✍️ **Comentario**: Haz doble clic en este bloque y complementa el gráfico con un breve texto que explique qué estamos viendo.

Comienza a escribir debajo de este texto, una vez escritas tus conclusiones, **elimina estas instrucciones** (de aqui hacia arriba de este bloque) para dejar solamente tus hallazgos.

## 🧩 Paso 7. Insight Ejecutivo para Stakeholders

Se traducen los hallazgos del análisis del experimento A/B en conclusiones accionables para el negocio, enfocadas en **versión de página, conversión, gasto promedio, canales de tráfico y tipo de usuario**.

**Preguntas a responder:**  
- ¿Qué página genera mayor conversión y gasto promedio?  
- ¿Qué canales de tráfico son más efectivos para generar conversiones?  
- ¿Existen diferencias significativas según el tipo de usuario?  
- ¿Qué recomendaciones se pueden tomar para optimizar la estrategia de marketing?


---

### 🌟 Insight Ejecutivo basado en el Experimento A/B

#### 🔍 **Comparación de página (A vs B)**  

**Gasto promedio por usuario que convirtió:**
- Observacion 1 aquí
- Observacion 2 aquí
- **Interpretación:**

<br>

**Tasa de conversión:** 
- Observacion 1 aquí
- Observacion 2 aquí
- **Interpretación:**

---

#### 📊 **Segmentación por fuente de tráfico**
- Observacion aquí
- **Interpretación:**
 
 ---

#### 📊 **Segmentación por tipo de usuario**
- Observacion aquí
- **Interpretación:**

---

Las visualizaciones usadas respaldan los resultados estadísticos de pasos anteriores.

---

#### 💡 **Recomendaciones de negocio:** 
- Recomendación aquí
-  Recomendación aquí